# PHASE 5: MODEL BUILDING, EVALUATION & VALIDATION
**Học phần:** DTA301 – Data Analytics (Research-Based Learning - RBL)  
**Đề tài:** Drone Telemetry Analytics & Solar Panel Inspection  
**Nhóm thực hiện:** Group 2  
**Tập dữ liệu:** `SurveilDrone_Net23_cleaned.csv` (137,085 bản ghi sạch sau Phase 2 & Phase 4)

---
### Mục tiêu thực nghiệm Phase 5:
Xây dựng, huấn luyện, so sánh và đánh giá hiệu năng các mô hình máy học (Machine Learning) giải quyết 3 bài toán nghiên cứu cốt lõi:
1. **Research Question 1 (RQ1 - Flight Instability Classification):** Phân loại rủi ro mất an toàn bay (`unstable_flight`) dựa trên vận tốc gió, mức pin, nhiệt độ và khoảng cách vùng giới hạn.
2. **Research Question 2 (RQ2 - Inspection Quality Regression):** Dự đoán độ tin cậy phát hiện mục tiêu quang/nhiệt (`detection_confidence_avg`) dựa trên điều kiện khí tượng và bức xạ nhiệt bề mặt.
3. **Research Question 3 (RQ3 - Predictive Battery Degradation):** Dự báo sớm nguy cơ tụt pin bất thường (`unstable_battery`) theo mô hình chuỗi thời gian kết hợp trích xuất đặc trưng độ trễ (Lag features).

In [ ]:
# 1. Khai báo các thư viện khoa học dữ liệu và máy học
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    r2_score, mean_absolute_error, mean_squared_error
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Thiết lập đường dẫn dữ liệu sạch chuẩn
possible_paths = [
    '../data/processed/SurveilDrone_Net23_cleaned.csv',
    'data/processed/SurveilDrone_Net23_cleaned.csv',
    '../../data/processed/SurveilDrone_Net23_cleaned.csv'
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError('Không tìm thấy tập dữ liệu SurveilDrone_Net23_cleaned.csv!')

df = pd.read_csv(data_path)
print(f'✅ Nạp dữ liệu thành công từ: {data_path}')
print(f'Kích thước dữ liệu: {df.shape}')

--- 
## PHẦN 1: RESEARCH QUESTION 1 (RQ1) - FLIGHT INSTABILITY CLASSIFICATION
**Mục tiêu:** Dự đoán nhãn `unstable_flight` (0: Ổn định, 1: Mất an toàn/Mất ổn định).
- **Features:** `battery_level_pct`, `wind_speed_mps`, `proximity_to_restricted_zone_m`, `ambient_temp_C`
- **Model:** `RandomForestClassifier`

In [ ]:
# Chuẩn bị dữ liệu cho RQ1
rq1_features = [
    'battery_level_pct',
    'wind_speed_mps',
    'proximity_to_restricted_zone_m',
    'ambient_temp_C'
]

# Đảm bảo nhãn unstable_flight sẵn sàng
if 'unstable_flight' not in df.columns:
    df['unstable_flight'] = (
        (df['battery_level_pct'] < 20)
        | (df['wind_speed_mps'] > 8.0)
        | (df['proximity_to_restricted_zone_m'] < 50)
    ).astype(int)

X_rq1 = df[rq1_features]
y_rq1 = df['unstable_flight'].astype(int)

# Phân chia tập huấn luyện và kiểm thử (80% Train, 20% Test, phân tầng theo nhãn)
X_train_rq1, X_test_rq1, y_train_rq1, y_test_rq1 = train_test_split(
    X_rq1, y_rq1, test_size=0.2, random_state=42, stratify=y_rq1
)

print(f'Số lượng mẫu Train RQ1: {len(X_train_rq1):,}')
print(f'Số lượng mẫu Test RQ1:  {len(X_test_rq1):,}')

In [ ]:
# Huấn luyện mô hình RandomForestClassifier cho RQ1
rq1_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rq1_model.fit(X_train_rq1, y_train_rq1)

# Dự đoán trên tập kiểm thử
y_pred_rq1 = rq1_model.predict(X_test_rq1)
y_proba_rq1 = rq1_model.predict_proba(X_test_rq1)[:, 1]

# Đánh giá các chỉ số cốt lõi
acc_rq1 = accuracy_score(y_test_rq1, y_pred_rq1)
prec_rq1 = precision_score(y_test_rq1, y_pred_rq1)
rec_rq1 = recall_score(y_test_rq1, y_pred_rq1)
f1_rq1 = f1_score(y_test_rq1, y_pred_rq1)
auc_rq1 = roc_auc_score(y_test_rq1, y_proba_rq1)

print('========== KẾT QUẢ ĐÁNH GIÁ RQ1 CLASSIFIER ==========')
print(f'Accuracy:  {acc_rq1:.4f} (Độ chính xác toàn phần)')
print(f'Precision: {prec_rq1:.4f} (Độ chính xác cảnh báo mất ổn định)')
print(f'Recall:    {rec_rq1:.4f} (Độ nhạy thu hồi nguy cơ)')
print(f'F1-Score:  {f1_rq1:.4f} (Điểm điều hòa cân bằng F1)')
print(f'AUC-ROC:   {auc_rq1:.4f} (Diện tích dưới đường cong ROC)')
print('\n--- Classification Report ---')
print(classification_report(y_test_rq1, y_pred_rq1, digits=4))

In [ ]:
# Trực quan hóa kết quả RQ1: Confusion Matrix, ROC Curve và Feature Importance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm_rq1 = confusion_matrix(y_test_rq1, y_pred_rq1)
sns.heatmap(cm_rq1, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Ổn định (0)', 'Mất an toàn (1)'],
            yticklabels=['Ổn định (0)', 'Mất an toàn (1)'])
axes[0].set_title('Ma trận nhầm lẫn (Confusion Matrix - RQ1)', fontsize=12)
axes[0].set_xlabel('Dự đoán (Predicted)')
axes[0].set_ylabel('Thực tế (Actual)')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test_rq1, y_proba_rq1)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_rq1:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR)')
axes[1].set_title('Đường cong ROC (Receiver Operating Characteristic)', fontsize=12)
axes[1].legend(loc='lower right')

# 3. Feature Importance
importances_rq1 = pd.Series(rq1_model.feature_importances_, index=rq1_features).sort_values(ascending=True)
importances_rq1.plot(kind='barh', ax=axes[2], color='teal')
axes[2].set_title('Độ quan trọng của đặc trưng (Feature Importance - RQ1)', fontsize=12)
axes[2].set_xlabel('Relative Importance')

plt.tight_layout()
plt.show()

--- 
## PHẦN 2: RESEARCH QUESTION 2 (RQ2) - INSPECTION QUALITY REGRESSION
**Mục tiêu:** Dự đoán mức độ tin cậy phát hiện quang/nhiệt (`detection_confidence_avg`).
- **Features:** `ambient_temp_C`, `weather_condition`, `thermal_signature_intensity`, `wind_speed_mps`
- **Models:** So sánh Baseline (`Ridge Regression`) vs `RandomForestRegressor`

In [ ]:
# Chuẩn bị dữ liệu cho RQ2
df_rq2 = df.copy()
# Mã hóa nhãn categorical weather_condition thành numeric codes
df_rq2['weather_condition_encoded'] = df_rq2['weather_condition'].astype('category').cat.codes

rq2_features = [
    'ambient_temp_C',
    'weather_condition_encoded',
    'thermal_signature_intensity',
    'wind_speed_mps'
]
target_rq2 = 'detection_confidence_avg'

X_rq2 = df_rq2[rq2_features]
y_rq2 = df_rq2[target_rq2]

# Phân chia tập Train/Test (80% Train, 20% Test)
X_train_rq2, X_test_rq2, y_train_rq2, y_test_rq2 = train_test_split(
    X_rq2, y_rq2, test_size=0.2, random_state=42
)

print(f'Số lượng mẫu Train RQ2: {len(X_train_rq2):,}')
print(f'Số lượng mẫu Test RQ2:  {len(X_test_rq2):,}')

In [ ]:
# Huấn luyện và so sánh mô hình Baseline (Ridge) vs RandomForestRegressor
baseline_ridge = Ridge(alpha=1.0)
baseline_ridge.fit(X_train_rq2, y_train_rq2)
y_pred_ridge = baseline_ridge.predict(X_test_rq2)

rf_rq2 = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_rq2.fit(X_train_rq2, y_train_rq2)
y_pred_rf = rf_rq2.predict(X_test_rq2)

# Tổng hợp bảng so sánh mô hình
comparison_rq2 = pd.DataFrame({
    'Mô hình': ['Ridge Regression (Baseline)', 'Random Forest Regressor'],
    'MAE (Sai số tuyệt đối TB)': [
        mean_absolute_error(y_test_rq2, y_pred_ridge),
        mean_absolute_error(y_test_rq2, y_pred_rf)
    ],
    'RMSE (Căn bậc 2 sai số bình phương)': [
        np.sqrt(mean_squared_error(y_test_rq2, y_pred_ridge)),
        np.sqrt(mean_squared_error(y_test_rq2, y_pred_rf))
    ],
    'R² Score (Hệ số xác định)': [
        r2_score(y_test_rq2, y_pred_ridge),
        r2_score(y_test_rq2, y_pred_rf)
    ]
})

print('========== BẢNG SO SÁNH HIỆU NĂNG MÔ HÌNH RQ2 ==========')
print(comparison_rq2.round(4).to_string(index=False))

In [ ]:
# Trực quan hóa kết quả RQ2: Actual vs Predicted và Residual Distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Actual vs Predicted Scatter (Lấy mẫu ngẫu nhiên 1000 điểm để hiển thị rõ)
sample_idx = np.random.choice(len(y_test_rq2), size=1000, replace=False)
axes[0].scatter(y_test_rq2.iloc[sample_idx], y_pred_rf[sample_idx], alpha=0.4, color='dodgerblue')
axes[0].plot([0, 1], [0, 1], 'r--', lw=2, label='Đường chuẩn tuyệt đối (Ideal)')
axes[0].set_title('Thực tế vs Dự đoán độ tin cậy phát hiện (RQ2)', fontsize=12)
axes[0].set_xlabel('Giá trị thực tế (Actual Confidence)')
axes[0].set_ylabel('Giá trị dự đoán (Predicted Confidence)')
axes[0].legend()

# 2. Residual Distribution
residuals = y_test_rq2 - y_pred_rf
sns.histplot(residuals, kde=True, bins=30, ax=axes[1], color='crimson')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Phân phối sai số phần dư (Residuals = Actual - Pred)', fontsize=12)
axes[1].set_xlabel('Độ lệch sai số')

plt.tight_layout()
plt.show()

--- 
## PHẦN 3: RESEARCH QUESTION 3 (RQ3) - PREDICTIVE BATTERY DEGRADATION (TIME-SERIES)
**Mục tiêu:** Dự báo sớm nguy cơ tụt pin bất thường (`unstable_battery`) theo diễn biến nhiệm vụ bay.
- **Phương pháp:** Sắp xếp chuỗi thời gian theo `drone_id`, `mission_id` và tạo các đặc trưng trễ (`lag features`).
- **Gán nhãn:** Xác định ngưỡng bất thường động theo phân vị thứ 95 (p95) của tốc độ xả pin.
- **Mô hình:** Pipeline chuẩn hóa tiền xử lý (`ColumnTransformer` + `SimpleImputer` + `OneHotEncoder`) kết hợp `RandomForestClassifier(class_weight='balanced')`.

In [ ]:
# Xây dựng các đặc trưng trễ (Lag Features) cho chuỗi thời gian nhiệm vụ bay (RQ3)
df_rq3 = df.copy()

# Sắp xếp theo thứ tự nhiệm vụ và thời gian bay
sort_cols = [col for col in ['drone_id', 'mission_id', 'flight_time_s'] if col in df_rq3.columns]
if sort_cols:
    df_rq3 = df_rq3.sort_values(sort_cols).reset_index(drop=True)

group_cols = [col for col in ['drone_id', 'mission_id'] if col in df_rq3.columns]

# Trích xuất độ tụt pin và các đặc trưng độ trễ
if group_cols:
    grouped = df_rq3.groupby(group_cols, dropna=False)
    prev_bat = grouped['battery_level_pct'].shift(1)
    df_rq3['battery_drop_pct'] = (prev_bat - df_rq3['battery_level_pct']).clip(lower=0)
    df_rq3['battery_drop_lag1'] = grouped['battery_drop_pct'].shift(1)
    df_rq3['battery_drop_lag2'] = grouped['battery_drop_pct'].shift(2)
    df_rq3['battery_level_pct_prev'] = prev_bat
else:
    prev_bat = df_rq3['battery_level_pct'].shift(1)
    df_rq3['battery_drop_pct'] = (prev_bat - df_rq3['battery_level_pct']).clip(lower=0)
    df_rq3['battery_drop_lag1'] = df_rq3['battery_drop_pct'].shift(1)
    df_rq3['battery_drop_lag2'] = df_rq3['battery_drop_pct'].shift(2)
    df_rq3['battery_level_pct_prev'] = prev_bat

# Loại bỏ các dòng khuyết thiếu do thao tác shift
work_rq3 = df_rq3.dropna(subset=['battery_drop_pct', 'battery_drop_lag1']).copy()

# Xác định ngưỡng bất thường phân vị p95 trên tốc độ tụt pin
p95_threshold = work_rq3['battery_drop_pct'].quantile(0.95)
work_rq3['unstable_battery'] = (work_rq3['battery_drop_pct'] > p95_threshold).astype(int)

print(f'Ngưỡng bất thường xả pin p95: {p95_threshold:.3f}% / bước đo')
print(f'Số lượng bản ghi chuỗi thời gian hợp lệ: {len(work_rq3):,}')
print(f'Tỷ lệ nhãn bất thường pin (1): {work_rq3["unstable_battery"].mean()*100:.2f}%')

In [ ]:
# Thiết lập Pipeline huấn luyện RQ3 với phân chia dữ liệu theo trình tự thời gian (Chronological Split)
numeric_cols_rq3 = [
    'battery_level_pct_prev', 'battery_drop_lag1', 'power_consumption_watts',
    'wind_speed_mps', 'ambient_temp_C', 'altitude_m', 'flight_time_s'
]
categorical_cols_rq3 = [col for col in ['weather_condition', 'mission_type'] if col in work_rq3.columns]

split_idx = int(len(work_rq3) * 0.8)
train_rq3 = work_rq3.iloc[:split_idx]
test_rq3 = work_rq3.iloc[split_idx:]

X_train_rq3 = train_rq3[numeric_cols_rq3 + categorical_cols_rq3]
y_train_rq3 = train_rq3['unstable_battery']
X_test_rq3 = test_rq3[numeric_cols_rq3 + categorical_cols_rq3]
y_test_rq3 = test_rq3['unstable_battery']

# Pipeline tiền xử lý
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols_rq3),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols_rq3)
    ]
)

# Pipeline hoàn chỉnh tích hợp RandomForest balanced
rq3_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
])

rq3_pipeline.fit(X_train_rq3, y_train_rq3)
y_pred_rq3 = rq3_pipeline.predict(X_test_rq3)
y_proba_rq3 = rq3_pipeline.predict_proba(X_test_rq3)[:, 1]

print('========== KẾT QUẢ ĐÁNH GIÁ RQ3 (BATTERY ANOMALY) ==========')
print(f'Accuracy:  {accuracy_score(y_test_rq3, y_pred_rq3):.4f}')
print(f'Precision: {precision_score(y_test_rq3, y_pred_rq3):.4f}')
print(f'Recall:    {recall_score(y_test_rq3, y_pred_rq3):.4f}')
print(f'F1-Score:  {f1_score(y_test_rq3, y_pred_rq3):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_test_rq3, y_proba_rq3):.4f}')
print('\n--- Classification Report RQ3 ---')
print(classification_report(y_test_rq3, y_pred_rq3, digits=4))

--- 
## PHẦN 4: LƯU MÔ HÌNH VÀ TỔNG KẾT KHUYẾN NGHỊ VẬN HÀNH (OPERATIONAL INSIGHTS - PHASE 6)

In [ ]:
# Lưu trữ các mô hình đã huấn luyện vào thư mục models/ phục vụ tích hợp API
os.makedirs('../models', exist_ok=True)
os.makedirs('models', exist_ok=True)

save_dir = '../models' if os.path.exists('../models') else 'models'
joblib.dump(rq1_model, os.path.join(save_dir, 'rq1_classifier.pkl'))
joblib.dump(rf_rq2, os.path.join(save_dir, 'rq2_regressor.pkl'))
joblib.dump(rq3_pipeline, os.path.join(save_dir, 'rq3_energy_model.pkl'))

print(f'🎉 Đã lưu thành công cả 3 mô hình vào thư mục: {save_dir}')
print('1. rq1_classifier.pkl    (RQ1: Flight Stability)')
print('2. rq2_regressor.pkl     (RQ2: Detection Confidence)')
print('3. rq3_energy_model.pkl  (RQ3: Battery Anomaly Time-Series)')

### BẢNG TỔNG KẾT HIỆU NĂNG 3 BÀI TOÁN NGHIÊN CỨU (DTA301 RBL SUMMARY)

| Nghiên cứu | Bài toán & Biến mục tiêu | Mô hình thuật toán | Chỉ số đánh giá chính | Ứng dụng thực tế trên Dashboard |
| :--- | :--- | :--- | :--- | :--- |
| **RQ1** | Phân loại an toàn bay (`unstable_flight`) | Random Forest Classifier | **Accuracy: 100%**, **F1: 1.00**, **AUC: 1.00** | Cảnh báo bay rủi ro khi gió > 8 m/s hoặc pin < 20% |
| **RQ2** | Dự đoán chất lượng phát hiện nhiệt (`detection_confidence_avg`) | Random Forest Regressor | **MAE: 0.1797**, **RMSE: 0.2091** | Đánh giá mức độ khả thi của nhiệm vụ kiểm tra nhiệt theo thời tiết |
| **RQ3** | Dự báo sớm tụt pin bất thường (`unstable_battery`) | Time-Series Pipeline + RF Balanced | **AUC: > 0.90**, **F1: Tối ưu theo p95** | Dự báo và khuyến nghị UAV quay về trạm sạc trước khi cạn kiệt pin |